02_feature_engineering_model.ipynb

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load from parquet — much faster than re-reading all .psv files
df = pd.read_parquet('../data/combined_dataset.parquet')

print(f"✅ Dataset loaded: {df.shape[0]:,} rows | {df['PatientID'].nunique():,} patients")
print(f"   Columns: {df.shape[1]}")

✅ Dataset loaded: 1,552,210 rows | 40,336 patients
   Columns: 43


Cell 2 — Select the 8 Key Clinical Features

In [2]:
# These are the clinically validated sepsis indicators
# Matching exactly what published papers use
FEATURES = [
    'HR',      # Heart rate
    'Temp',    # Temperature
    'Resp',    # Respiratory rate
    'O2Sat',   # Oxygen saturation
    'SBP',     # Systolic blood pressure
    'MAP',     # Mean arterial pressure
    'WBC',     # White blood cell count
    'Lactate', # Lactate (most important sepsis marker)
]

TARGET = 'SepsisLabel'

# Check which features exist in our dataset
available = [f for f in FEATURES if f in df.columns]
missing_features = [f for f in FEATURES if f not in df.columns]

print(f"✅ Features available : {available}")
print(f"⚠️  Features missing  : {missing_features}")
print(f"\nTarget column exists : {TARGET in df.columns}")

✅ Features available : ['HR', 'Temp', 'Resp', 'O2Sat', 'SBP', 'MAP', 'WBC', 'Lactate']
⚠️  Features missing  : []

Target column exists : True


Cell 3 — Handle Missing Values

In [3]:
# Work only with our selected features + identifiers
cols_needed = ['PatientID', 'Hour', TARGET] + available
df_clean = df[cols_needed].copy()

print("Missing values BEFORE imputation:")
print((df_clean[available].isnull().sum() / len(df_clean) * 100).round(1).to_string())

# Forward fill within each patient (vitals carry forward)
df_clean[available] = df_clean.groupby('PatientID')[available].transform(
    lambda x: x.fillna(method='ffill')
)

# Fill remaining with column median (for first hours with no readings)
for col in available:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)

print("\n✅ Missing values AFTER imputation:")
print((df_clean[available].isnull().sum() / len(df_clean) * 100).round(1).to_string())
print("\n✅ All missing values handled")

Missing values BEFORE imputation:
HR          9.9
Temp       66.2
Resp       15.4
O2Sat      13.1
SBP        14.6
MAP        12.5
WBC        93.6
Lactate    97.3

✅ Missing values AFTER imputation:
HR         0.0
Temp       0.0
Resp       0.0
O2Sat      0.0
SBP        0.0
MAP        0.0
WBC        0.0
Lactate    0.0

✅ All missing values handled


Cell 4 — Create the 6-Hour Prediction Label

In [4]:
# Y = 1 if patient develops sepsis in the NEXT 6 hours
# This is the core of the project — predicting BEFORE it happens

def create_prediction_label(group, hours_ahead=6):
    """For each hour, check if sepsis occurs in next 6 hours."""
    label = group[TARGET].values.copy()
    prediction = np.zeros(len(label), dtype=int)

    for i in range(len(label)):
        # Look forward up to 6 hours
        future_window = label[i : i + hours_ahead]
        if future_window.sum() > 0:
            prediction[i] = 1

    return pd.Series(prediction, index=group.index)

print("Creating 6-hour prediction labels...")
print("This takes 2-3 minutes — please wait...")

df_clean['Label_6h'] = df_clean.groupby('PatientID', group_keys=False).apply(
    create_prediction_label
)

# Summary
total_positive = df_clean['Label_6h'].sum()
total = len(df_clean)
print(f"\n✅ Labels created")
print(f"   Positive (sepsis risk) : {total_positive:,} rows ({total_positive/total*100:.1f}%)")
print(f"   Negative (safe)        : {total-total_positive:,} rows ({(total-total_positive)/total*100:.1f}%)")

Creating 6-hour prediction labels...
This takes 2-3 minutes — please wait...

✅ Labels created
   Positive (sepsis risk) : 39,796 rows (2.6%)
   Negative (safe)        : 1,512,414 rows (97.4%)


Cell 5 — Build Feature Matrix

In [5]:
from sklearn.preprocessing import StandardScaler

# Final feature matrix
X = df_clean[available].values
y = df_clean['Label_6h'].values

print(f"Feature matrix X shape : {X.shape}")
print(f"Target vector y shape  : {y.shape}")
print(f"Positive class rate    : {y.mean()*100:.1f}%")

Feature matrix X shape : (1552210, 8)
Target vector y shape  : (1552210,)
Positive class rate    : 2.6%


Cell 6 — Train/Test Split + SMOTE

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Split BEFORE applying SMOTE — critical rule
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Train size : {X_train.shape[0]:,} rows")
print(f"Test size  : {X_test.shape[0]:,} rows")
print(f"\nClass distribution BEFORE SMOTE:")
print(f"  Train — Positive: {y_train.sum():,} | Negative: {(y_train==0).sum():,}")

# Apply SMOTE only on training data
print("\nApplying SMOTE... (takes ~1 minute)")
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"\n✅ Class distribution AFTER SMOTE:")
print(f"  Train — Positive: {y_train_sm.sum():,} | Negative: {(y_train_sm==0).sum():,}")
print(f"  Train size now   : {X_train_sm.shape[0]:,} rows")

# Normalize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled  = scaler.transform(X_test)

print("\n✅ Features normalized with StandardScaler")

Train size : 1,241,768 rows
Test size  : 310,442 rows

Class distribution BEFORE SMOTE:
  Train — Positive: 31,837 | Negative: 1,209,931

Applying SMOTE... (takes ~1 minute)

✅ Class distribution AFTER SMOTE:
  Train — Positive: 1,209,931 | Negative: 1,209,931
  Train size now   : 2,419,862 rows

✅ Features normalized with StandardScaler


Cell 7 — Train Logistic Regression (Baseline)

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

print("Training Logistic Regression baseline...")

lr = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr.fit(X_train_scaled, y_train_sm)

lr_proba = lr.predict_proba(X_test_scaled)[:, 1]
lr_pred  = lr.predict(X_test_scaled)
lr_auroc = roc_auc_score(y_test, lr_proba)

print(f"\n✅ LOGISTIC REGRESSION RESULTS")
print(f"   AUROC : {lr_auroc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, lr_pred,
      target_names=['No Sepsis Risk', 'Sepsis Risk']))

Training Logistic Regression baseline...

✅ LOGISTIC REGRESSION RESULTS
   AUROC : 0.6510

Classification Report:
                precision    recall  f1-score   support

No Sepsis Risk       0.98      0.64      0.77    302483
   Sepsis Risk       0.04      0.59      0.08      7959

      accuracy                           0.64    310442
     macro avg       0.51      0.61      0.43    310442
  weighted avg       0.96      0.64      0.76    310442



Cell 8 — Train XGBoost (Main Model)

In [9]:
from xgboost import XGBClassifier

print("Training XGBoost... (takes 2-3 minutes)")

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

xgb.fit(
    X_train_sm, y_train_sm,
    eval_set=[(X_test, y_test)],
    verbose=50
)

xgb_proba = xgb.predict_proba(X_test)[:, 1]
xgb_pred  = xgb.predict(X_test)
xgb_auroc = roc_auc_score(y_test, xgb_proba)

print(f"\n✅ XGBOOST RESULTS")
print(f"   AUROC : {xgb_auroc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, xgb_pred,
      target_names=['No Sepsis Risk', 'Sepsis Risk']))

Training XGBoost... (takes 2-3 minutes)
[0]	validation_0-logloss:0.67878
[50]	validation_0-logloss:0.43645
[100]	validation_0-logloss:0.32352
[150]	validation_0-logloss:0.28539
[200]	validation_0-logloss:0.26282
[250]	validation_0-logloss:0.24357
[299]	validation_0-logloss:0.22916

✅ XGBOOST RESULTS
   AUROC : 0.7097

Classification Report:
                precision    recall  f1-score   support

No Sepsis Risk       0.98      0.97      0.97    302483
   Sepsis Risk       0.08      0.12      0.10      7959

      accuracy                           0.94    310442
     macro avg       0.53      0.54      0.53    310442
  weighted avg       0.95      0.94      0.95    310442



Cell 9 — Compare Models + Save Results

In [10]:
from sklearn.metrics import f1_score, precision_score, recall_score
import joblib

# Build comparison table
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'XGBoost'],
    'AUROC':  [round(lr_auroc, 4),  round(xgb_auroc, 4)],
    'F1':     [round(f1_score(y_test, lr_pred), 4),
               round(f1_score(y_test, xgb_pred), 4)],
    'Precision': [round(precision_score(y_test, lr_pred), 4),
                  round(precision_score(y_test, xgb_pred), 4)],
    'Recall': [round(recall_score(y_test, lr_pred), 4),
               round(recall_score(y_test, xgb_pred), 4)],
})

print("MODEL COMPARISON:")
print(results.to_string(index=False))

# Save models
joblib.dump(lr,  '../models/logistic_regression.pkl')
joblib.dump(xgb, '../models/xgboost_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

# Save results
results.to_csv('../outputs/02_model_comparison.csv', index=False)

print(f"\n✅ Models saved to models/")
print(f"✅ Results saved to outputs/")

# Final verdict
print(f"\n{'='*40}")
print(f"WEEK 2 CHECKPOINT")
print(f"{'='*40}")
print(f"Logistic Regression AUROC : {lr_auroc:.4f}  (target > 0.72)")
print(f"XGBoost AUROC             : {xgb_auroc:.4f}  (target > 0.82)")

if xgb_auroc >= 0.82:
    print(f"\n🔥 XGBoost TARGET HIT — Week 2 complete!")
elif xgb_auroc >= 0.75:
    print(f"\n✅ Good result — acceptable for Week 2")
else:
    print(f"\n⚠️  AUROC below target — paste results here for fix")
print(f"{'='*40}")

MODEL COMPARISON:
              Model  AUROC     F1  Precision  Recall
Logistic Regression 0.6510 0.0768     0.0411  0.5891
            XGBoost 0.7097 0.0970     0.0837  0.1152

✅ Models saved to models/
✅ Results saved to outputs/

WEEK 2 CHECKPOINT
Logistic Regression AUROC : 0.6510  (target > 0.72)
XGBoost AUROC             : 0.7097  (target > 0.82)

⚠️  AUROC below target — paste results here for fix
